# Model Evaluation — Isolation Forest
Visual analysis of model performance and anomaly score distribution.

In [ ]:
import glob

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, auc, confusion_matrix, roc_curve

from src.detector.features import select_features

sns.set_theme(style="whitegrid")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [ ]:
# Load dataset
files = glob.glob("data/raw/metrics_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["is_anomaly"] = (df["label"] == "anomaly").astype(int)
print(f"Dataset: {len(df)} rows | Anomaly rate: {df['is_anomaly'].mean():.1%}")

In [ ]:
# Load model and scaler
model = mlflow.sklearn.load_model("models:/k8s-anomaly-detector/latest")
scaler = joblib.load("src/models/scaler.joblib")
X = select_features(df)
X_scaled = scaler.transform(X.values)
y = df["is_anomaly"].values
anomaly_scores = -model.score_samples(X_scaled)
predictions = np.where(model.predict(X_scaled) == -1, 1, 0)
print("Model loaded successfully")

In [ ]:
# Plot 1: Anomaly score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    anomaly_scores[y == 0],
    bins=40,
    alpha=0.7,
    label="Normal",
    color="steelblue",
)
axes[0].hist(
    anomaly_scores[y == 1],
    bins=40,
    alpha=0.7,
    label="Anomaly",
    color="crimson",
)
axes[0].set_title("Anomaly score distribution")
axes[0].set_xlabel("Anomaly score (higher = more anomalous)")
axes[0].legend()

# Plot 2: ROC curve
fpr, tpr, _ = roc_curve(y, anomaly_scores)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_title("ROC curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.savefig("docs/model_roc_scores.png", dpi=150)
plt.show()

In [ ]:
# Plot 3: Anomaly score over time — the money plot
df["anomaly_score"] = anomaly_scores
df["detected"] = predictions

plt.figure(figsize=(16, 5))
for pod, group in df.groupby("pod"):
    plt.plot(
        group["timestamp"], group["anomaly_score"], alpha=0.7, label=pod.split("-")[0]
    )

plt.axhline(y=0.5, color="red", linestyle="--", lw=1.5, label="Threshold (0.5)")
anomaly_periods = df[df["is_anomaly"] == 1]
if len(anomaly_periods) > 0:
    plt.axvspan(
        anomaly_periods["timestamp"].min(),
        anomaly_periods["timestamp"].max(),
        alpha=0.15,
        color="red",
        label="Injected anomaly window",
    )

plt.title("Anomaly score over time by pod")
plt.ylabel("Anomaly score")
plt.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig("docs/model_scores_timeline.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Anomaly"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion matrix")
plt.tight_layout()
plt.savefig("docs/confusion_matrix.png", dpi=150)
plt.show()